In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

BASE = Path("..")

INPUT_FILES = {
    "usernames": BASE / "dataset_filtered/usernames_filtered.jsonl",
    "phone_numbers": BASE / "dataset_filtered/numbers_filtered.jsonl"
}

OUTPUT_DIR = BASE / "dataset_normalized"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILES = {
    "usernames": OUTPUT_DIR / "usernames_normalized.jsonl",
    "phone_numbers": OUTPUT_DIR / "numbers_normalized.jsonl"
}

In [2]:
KEY_MAP = {
    "n": "nft",
    "u": "asset_name",
    "c": "categories",
    "ly": "layer",
    "oa": "owner_address",
    "aid": "action_id",
    "tid": "trace_id",
    "t": "timestamp",
    "et": "end_timestamp",
    "slt": "start_lt",
    "elt": "end_lt",
    "y": "action_type",
    "ok": "success",
    "det": "details"
}

In [3]:
def convert_timestamp(value):
    if value is None or value == "":
        return None

    try:
        return datetime.fromtimestamp(
            int(value),
            tz=timezone.utc
        ).isoformat(timespec="seconds").replace("+00:00", "Z")
    except (ValueError, TypeError, OverflowError, OSError):
        return None

In [4]:
def normalizza_jsonl(input_path, output_path):
    stats = Counter()

    with open(input_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line in fin:
            if not line.strip():
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                stats["json_non_validi"] += 1
                continue

            # Rinomina colonne
            normalized = {
                KEY_MAP.get(key, key): value
                for key, value in record.items()
            }

            # Converte timestamp
            for col in ["timestamp", "end_timestamp"]:
                if col in normalized:
                    original = normalized[col]
                    converted = convert_timestamp(original)

                    if converted is None and original is not None:
                        stats["timestamp_non_validi"] += 1

                    normalized[col] = converted

            fout.write(
                json.dumps(normalized, ensure_ascii=False) + "\n"
            )

            stats["righe_convertite"] += 1

    print(f"\nFILE: {input_path.name}")
    print(f"Righe convertite: {stats['righe_convertite']:,}")
    print(f"JSON non validi: {stats['json_non_validi']:,}")
    print(f"Timestamp non validi: {stats['timestamp_non_validi']:,}")
    print(
        "Dimensione output:",
        round(output_path.stat().st_size / 1024**2, 2),
        "MB"
    )

    return stats

In [5]:
stats_usernames = normalizza_jsonl(
    INPUT_FILES["usernames"],
    OUTPUT_FILES["usernames"]
)

stats_numbers = normalizza_jsonl(
    INPUT_FILES["phone_numbers"],
    OUTPUT_FILES["phone_numbers"]
)


FILE: usernames_filtered.jsonl
Righe convertite: 2,579,153
JSON non validi: 0
Timestamp non validi: 0
Dimensione output: 4003.0 MB

FILE: numbers_filtered.jsonl
Righe convertite: 837,146
JSON non validi: 0
Timestamp non validi: 0
Dimensione output: 1345.85 MB


In [6]:
import pandas as pd

df_usernames = pd.read_json(
    OUTPUT_FILES["usernames"],
    lines=True,
    nrows=5
)

df_numbers = pd.read_json(
    OUTPUT_FILES["phone_numbers"],
    lines=True,
    nrows=5
)

print("USERNAMES")
display(df_usernames)

print("PHONE NUMBERS")
display(df_numbers)

USERNAMES


,nft,asset_name,categories,layer,owner_address,action_id,trace_id,timestamp,end_timestamp,start_lt,end_lt,action_type,success,details
0,0:9c81d90cd3c57df4e0593e3493497894021c392cd011...,@wandou0007,[10 Letters],usernames,0:3fb7330d4ec1850d2b876295fc39467d335aa4db79bf...,iGzm0PwAjxwUjtW4U9JxaL2Aq+HGl/kqHfUekZGbcdw=,fUJfAk3EScZHAsS/+unO2heesled95K+2/9MDfjEkOY=,2025-08-12 08:20:45+00:00,2025-08-12T08:20:45Z,60384214000001,60384214000005,nft_transfer,True,{'nft_collection': '0:80D78A35F955A14B679FAA88...
1,0:9c81d90cd3c57df4e0593e3493497894021c392cd011...,@wandou0007,[10 Letters],usernames,0:3fb7330d4ec1850d2b876295fc39467d335aa4db79bf...,Pu3Ek5Jk8okUNDz7pJ79gYEli1X4tI3JlDMXFYWapdg=,OQZsthhxqpAgl46tMpayUzhjmj3pM6MKW5X+vUTq/6A=,2025-08-12 15:54:18+00:00,2025-08-12T15:54:18Z,60394900000002,60394900000006,nft_transfer,True,{'nft_collection': '0:80D78A35F955A14B679FAA88...
2,0:7f081d77ebbe434ed6bdea89a7298ba863a5ac162b72...,@list_control,[No_Categories],usernames,0:1b0c8da73ceb2a0381e6959939be37c32ad4206f389d...,IiANy1Dx3wps/mqpOX0HxVJGD09qxknGt1g8ZQC6aIA=,iTEM1ui/FlZcpxokNbSwY5IMi6UzS3hwVIZVZVmw9Bw=,2026-04-26 07:40:27+00:00,2026-04-26T07:40:27Z,72986662000001,72986662000005,nft_transfer,True,{'nft_collection': '0:80D78A35F955A14B679FAA88...
3,0:7f081d77ebbe434ed6bdea89a7298ba863a5ac162b72...,@list_control,[No_Categories],usernames,0:1b0c8da73ceb2a0381e6959939be37c32ad4206f389d...,a0nYK7cCU1s1JQ0MHVRqVzLylQGKjzvZ56x/1DXhF6o=,FAtkJKAUSRzNZazDr7fgeKKdcuY0+i+d9ZdYjhEZViI=,2026-06-03 07:51:25+00:00,2026-06-03T07:51:25Z,81237024000037,81237025000031,nft_transfer,True,{'nft_collection': '0:80D78A35F955A14B679FAA88...
4,0:7f081d77ebbe434ed6bdea89a7298ba863a5ac162b72...,@list_control,[No_Categories],usernames,0:1b0c8da73ceb2a0381e6959939be37c32ad4206f389d...,43FfrNkdVyLbDbM8ZDRFNfmU8+N9deP7EM0/phoGq1Y=,L645Y+tq1nRjmE2SMFXn9/nxSGR9L6yNVhy85yNCuZ4=,2026-07-26 14:46:28+00:00,2026-07-26T14:46:28Z,92692548000026,92692548000031,nft_transfer,True,{'nft_collection': '0:80D78A35F955A14B679FAA88...


PHONE NUMBERS


,nft,asset_name,categories,layer,owner_address,action_id,trace_id,timestamp,end_timestamp,start_lt,end_lt,action_type,success,details
0,0:97339c6c6c1022742e4446a2ef3ddfde83955f15f5ce...,+888 0342 4679,[No_Categories],phone_numbers,0:bce2384322fc65a2bbf552f5eda121da9a623319cb3f...,bXVVpY3GCc+sDOngAAaHBoWyNWiJLHa8e9Az6zgNH7Q=,hSjLk6GC8ubwJfh4OYHUt0MY7lL/ysvMBOpH+6VNbC4=,2022-12-13 18:41:43+00:00,2022-12-13T18:41:43Z,33631021000006,33631021000007,nft_transfer,True,{'nft_collection': '0:0E41DC1DC3C9067ED2424858...
1,0:97339c6c6c1022742e4446a2ef3ddfde83955f15f5ce...,+888 0342 4679,[No_Categories],phone_numbers,0:bce2384322fc65a2bbf552f5eda121da9a623319cb3f...,L0tllOXDPbMHebleS9KcFBT9+Gn+3KNUOQYlDxQPImg=,RRXyvi0+p/svhB9nWpHIakbhXyF8PtjzJJToRmE1Hew=,2022-12-13 18:43:44+00:00,2022-12-13T18:43:44Z,33631055000002,33631055000006,nft_transfer,True,{'nft_collection': '0:0E41DC1DC3C9067ED2424858...
2,0:97339c6c6c1022742e4446a2ef3ddfde83955f15f5ce...,+888 0342 4679,[No_Categories],phone_numbers,0:bce2384322fc65a2bbf552f5eda121da9a623319cb3f...,akC0zWYjEc0Mw1ByIOqPxUjBbFdMP+s+IVMPu80T4O8=,8+5MloAQHzZ97oRsaTqpEzuJ3vlE6VtXGTjP+1qjM+o=,2023-01-02 00:57:49+00:00,2023-01-02T00:57:49Z,34137590000002,34137590000007,nft_transfer,True,{'nft_collection': '0:0E41DC1DC3C9067ED2424858...
3,0:97339c6c6c1022742e4446a2ef3ddfde83955f15f5ce...,+888 0342 4679,[No_Categories],phone_numbers,0:bce2384322fc65a2bbf552f5eda121da9a623319cb3f...,COIxjQbgA5zaHHAVp6YX42GWpVClzOUjqeGsr/l0rj8=,AwR5rrsu+aR+t2f/sYh/srHTjMxn0HpHFsipaIDQmZg=,2023-01-02 21:11:01+00:00,2023-01-02T21:11:01Z,34159726000002,34159726000007,nft_transfer,True,{'nft_collection': '0:0E41DC1DC3C9067ED2424858...
4,0:97339c6c6c1022742e4446a2ef3ddfde83955f15f5ce...,+888 0342 4679,[No_Categories],phone_numbers,0:bce2384322fc65a2bbf552f5eda121da9a623319cb3f...,StZHQOko4T/BAsbtOAtzbDfms121h7T7Dh/a18Itiac=,xk6by+ibDvtv/ol/1Wmr3RiMxsaYIgscJIb72NR3qeA=,2023-11-06 10:08:06+00:00,2023-11-06T10:08:06Z,42306916000002,42306916000007,nft_transfer,True,{'nft_collection': '0:0E41DC1DC3C9067ED2424858...
